### trying to use the excel

In [1]:
import pandas as pd

In [2]:
x = pd.read_excel(r'..\disclosure_standards\raw\esrs_paragraph_mapping.xlsx', sheet_name="esrse1")

In [4]:
x.head()

,paragraph,details,nested1,nested2,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,1,The objective of this Standard is to specify D...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,(a),"how the undertaking affects climate change, in...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,(b),"the undertaking’s past, current, and future mi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,(c),the plans and capacity of the undertaking to a...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,(d),"any other actions taken by the undertaking, an...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
maybe_tables = x.groupby(["paragraph", "details"]).filter(lambda x: len(x) > 1)

tables = maybe_tables.groupby(["paragraph", "details"]).filter(lambda x: (x["nested1"].fillna('').str.len() > 3).all())

# tables.groupby(["paragraph", "details"]).agg(' '.join, axis=1)

tables.groupby(["paragraph", "details"]).agg(' '.join, axis=1)

TypeError: str.join() takes exactly one argument (0 given)

In [39]:
df = tables.groupby(["paragraph", "details"]).get_group(('AR 12.', '(d)'))

df.head()

,paragraph,details,nested1,nested2,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
216,AR 12.,(d),identified assets and business activities that...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
217,AR 12.,(d),Examples of climate-related transition events ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
218,AR 12.,(d),Policy and legal,Technology,Market,Reputation,NaN,NaN,NaN,NaN,NaN
219,AR 12.,(d),Increased pricing of GHG emissions,Substitution of existing products and services...,Changing customer behaviour,Shifts in consumer preferences,NaN,NaN,NaN,NaN,NaN
220,AR 12.,(d),Enhanced emissions- reporting obligations,Unsuccessful investment in new technologies,Uncertainty in market signals,Stigmatization of sector,NaN,NaN,NaN,NaN,NaN


In [34]:
' '.join(tables.iloc[:, 2:].astype(str).values.flatten())

'identified assets and business activities that are incompatible with or need significant efforts to be compatible with a transition to a climate-neutral economy (for example, due to significant locked-in GHG emissions or incompatibility with the requirements for Taxonomy-alignment under Commission Delegated Regulation (EU)\xa02021/2139). nan nan nan nan nan nan nan nan Examples of climate-related transition events (examples based on TCFD classification)  nan nan nan nan nan nan nan nan Policy and legal  Technology  Market  Reputation  nan nan nan nan nan Increased pricing of GHG emissions Substitution of existing products and services with lower emissions options Changing customer behaviour Shifts in consumer preferences nan nan nan nan nan Enhanced emissions- reporting obligations Unsuccessful investment in new technologies Uncertainty in market signals Stigmatization of sector nan nan nan nan nan Mandates on and regulation of existing products and services Costs of transition to low

In [35]:
rows = []
for _, row in tables.iloc[:, 2:].iterrows():
    row_text = ', '.join(
        f"{col}: {row[col]}"
        for col in tables.iloc[:, 2:].columns
        if pd.notna(row[col])
    )
    rows.append(row_text)

text = ' | '.join(rows)

In [36]:
text

'nested1: identified assets and business activities that are incompatible with or need significant efforts to be compatible with a transition to a climate-neutral economy (for example, due to significant locked-in GHG emissions or incompatibility with the requirements for Taxonomy-alignment under Commission Delegated Regulation (EU)\xa02021/2139). | nested1: Examples of climate-related transition events (examples based on TCFD classification)  | nested1: Policy and legal , nested2: Technology , Unnamed: 4: Market , Unnamed: 5: Reputation  | nested1: Increased pricing of GHG emissions, nested2: Substitution of existing products and services with lower emissions options, Unnamed: 4: Changing customer behaviour, Unnamed: 5: Shifts in consumer preferences | nested1: Enhanced emissions- reporting obligations, nested2: Unsuccessful investment in new technologies, Unnamed: 4: Uncertainty in market signals, Unnamed: 5: Stigmatization of sector | nested1: Mandates on and regulation of existing 

### Trying Docling

In [41]:
from docling.document_converter import DocumentConverter

In [43]:
converter = DocumentConverter()

doc = converter.convert(r"raw\ESRS Set 1.pdf").document

2025-12-15 23:03:25,365 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-12-15 23:03:28,233 - INFO - Going to convert document batch...
2025-12-15 23:03:28,236 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-12-15 23:03:28,254 - INFO - Loading plugin 'docling_defaults'
2025-12-15 23:03:28,259 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-12-15 23:03:28,274 - INFO - Loading plugin 'docling_defaults'
2025-12-15 23:03:28,287 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-12-15 23:03:28,425 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2025-12-15 23:03:28,426 - INFO - easyocr cannot be used because it is not installed.
2025-12-15 23:03:30,436 - INFO - Accelerator device: 'cuda:0'
[INFO] 2025-12-15 23:03:30,463 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-12-15 23:03:30,536 [RapidOCR] download_file.py:60: 

In [52]:
output_dir = r"raw/tables/"

for table_ix, table in enumerate(doc.tables):
    table_df: pd.DataFrame = table.export_to_dataframe(doc=doc)
    # print(f"## Table {table_ix}")
    # print(table_df.to_markdown())

    # Save the table as CSV
    element_csv_filename = output_dir + f"ESRS-table-{table_ix + 1}.csv"
    table_df.to_csv(element_csv_filename)


In [ ]:
doc.texts

[FormulaItem(self_ref='#/texts/0', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.FORMULA: 'formula'>, prov=[ProvenanceItem(page_no=147, bbox=BoundingBox(l=201.0, t=328.2534383138021, r=477.3333333333333, b=307.2534383138021, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 83))], orig='Numberof employeescoveredbycollective bargainingagreements X 100 Numberof employees', text='', formatting=None, hyperlink=None),
 TextItem(self_ref='#/texts/1', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, label=<DocItemLabel.PAGE_FOOTER: 'page_footer'>, prov=[ProvenanceItem(page_no=147, bbox=BoundingBox(l=173.66666666666666, t=46.25343831380212, r=458.0, b=37.25343831380212, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 67))], orig="Numberofemployeesworkinginestablishmentswithworkers'representatives", text="Numberofemployeesworkingi

: 

### Trying to automate it

In [ ]:
x = pd.read_html(r"..\disclosure_standards\raw\ESRS Set 1.html")

In [19]:
from bs4 import BeautifulSoup

def extract_oj_tables(html_path):
    # Load the HTML file
    with open(html_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    # Find all tables with class="oj-table"
    oj_tables = soup.find_all("table", class_="oj-table")

    dataframes = []
    for table in oj_tables:
        # Convert each table HTML into a DataFrame
        df = pd.read_html(str(table))[0]
        dataframes.append(df)

    return dataframes


In [60]:
x = extract_oj_tables(r"..\disclosure_standards\raw\ESRS Set 1.html")

C:\Users\krish\AppData\Local\Temp\ipykernel_14444\3167887275.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]
C:\Users\krish\AppData\Local\Temp\ipykernel_14444\3167887275.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]
C:\Users\krish\AppData\Local\Temp\ipykernel_14444\3167887275.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(table))[0]
C:\Users\krish\AppData\Local\Temp\ipykernel_14444\3167887275.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read fr

In [62]:
x[0]

,0,1,2,3
0,Topical ESRS,Sustainability matters covered in topical ESRS,Sustainability matters covered in topical ESRS,Sustainability matters covered in topical ESRS
1,NaN,Topic,Sub-topic,Sub-sub-topics
2,ESRS E1,Climate change,— Climate change adaptation — Climate chang...,NaN
3,—,Climate change adaptation,NaN,NaN
4,—,Climate change mitigation,NaN,NaN
...,...,...,...,...
129,—,Management of relationships with suppliers inc...,NaN,NaN
130,NaN,NaN,— Corruption and bribery,— Prevention and detection including training...
131,—,Corruption and bribery,NaN,NaN
132,—,Prevention and detection including training,NaN,NaN
